INSTALL LIBRARIES

In [1]:
!pip -q install pypdf sentence-transformers transformers accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 5.1 MB/s eta 0:00:00


IMPORT LIBRARIES

In [2]:
import re
import numpy as np
import torch

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


load pdf

In [3]:
PDF_PATH = "/content/Educational Loan Scheme.pdf"

reader = PdfReader(PDF_PATH)

pages = []

for page_number, page in enumerate(reader.pages):

    text = page.extract_text()

    if text:
        pages.append({
            "page": page_number + 1,
            "text": text
        })

print("Number of pages:", len(pages))

Number of pages: 2


IMPORT REQUIRED LIBRARIES

In [4]:
def clean_text(text):

    text = text.replace("\r", "\n")

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


for page in pages:
    page["text"] = clean_text(
        page["text"]
    )

print(pages[0]["text"][:2000])

Educational Loan Scheme 
Rajasthan Minority Finance & Development Co -operative Corporation extends 
educational loans with an objective to facilitate job oriented education for the 
eligible persons belonging to Minorities. Under the scheme a maximum amount of 
Rs. 20.00 Lakhs is available at the rate Rs. 4.00 lakhs p.a. for technical and 
professional courses of durations not exceeding five years. Further, for courses 
abroad, a maximum amount of Rs. 30.00 lakhs is available @ Rs. 6.00 lakhs per 
annum for a course durati on of maximum 5 years. The loan is payable in 
maximum five years after completion of the course. 
 
Target Group: 
The prime mandate of RMFDCC is to provide concessional finance to the Minorities 
for self employment/ income generation activities. As per the N ational Commission 
of Minorities Act, 1992 the notified Minorities are Muslims, Christians, Sikhs, 
Buddhists, and Parsis. Subsequently, Jains community was also added into the list 
of notified Minority Com

CLEAN EXTRACTED PDF TEXT

In [5]:
def create_section_chunks(pages):

    chunks = []

    chunk_id = 0

    for page in pages:

        lines = [
            line.strip()
            for line in page["text"].split("\n")
            if line.strip()
        ]

        current_heading = "General Information"
        current_content = []

        for line in lines:

            is_heading = (
                len(line) <= 80
                and line.endswith(":")
            )

            if is_heading:

                if current_content:

                    section_text = (
                        current_heading
                        + "\n"
                        + " ".join(current_content)
                    )

                    chunks.append({
                        "chunk_id": chunk_id,
                        "page": page["page"],
                        "heading": current_heading,
                        "text": section_text
                    })

                    chunk_id += 1

                current_heading = line.rstrip(":")
                current_content = []

            else:

                current_content.append(line)

        if current_content:

            section_text = (
                current_heading
                + "\n"
                + " ".join(current_content)
            )

            chunks.append({
                "chunk_id": chunk_id,
                "page": page["page"],
                "heading": current_heading,
                "text": section_text
            })

            chunk_id += 1

    return chunks


chunks = create_section_chunks(pages)

print("Total chunks:", len(chunks))

Total chunks: 10


LOAD MULTILINGUAL EMBEDDING MODEL

In [6]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/LaBSE"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print(
    "Multilingual embedding model loaded."
)

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.88GB            

... (7 outputs left out)

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 2.36MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

Multilingual embedding model loaded.


CREATE DOCUMENT EMBEDDINGS

In [7]:
document_texts = [
    chunk["text"]
    for chunk in chunks
]

document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(
    "Embedding shape:",
    document_embeddings.shape
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (10, 768)


INSTALL TRANSLATION SUPPORT


In [8]:
!pip -q install transformers sentencepiece

INSTALL TRANSLATION SUPPORT


In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

TRANSLATION_MODEL = "facebook/nllb-200-distilled-600M"

translation_tokenizer = AutoTokenizer.from_pretrained(
    TRANSLATION_MODEL,
    src_lang="mar_Deva"
)

translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANSLATION_MODEL
)

translation_model.eval()

print("NLLB translation model loaded successfully.")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB translation model loaded successfully.


IDENTIFY ENGLISH TARGET TOKEN

In [10]:
english_token_id = translation_tokenizer.get_vocab()["eng_Latn"]

print("English token ID:", english_token_id)

English token ID: 256047


MULTILINGUAL QUESTION TRANSLATION

In [11]:
# ============================================================
# CELL 21 - Multilingual Query Translation to English
# ============================================================

NLLB_LANGUAGE_CODES = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Marathi": "mar_Deva"
}

NIRUKTI_LANGUAGE_CODES = {
    "English": "eng_Latn",
    "Marwari": "mwr_Deva"
}


def translate_to_english(text, source_language):
    """
    Translate a user query into English.

    English, Hindi and Marathi:
        NLLB-200

    Marwari:
        Nirukti Marwari translation model
    """

    # --------------------------------------------------------
    # English
    # --------------------------------------------------------

    if source_language == "English":
        return text.strip()


    # --------------------------------------------------------
    # Marwari
    # --------------------------------------------------------

    if source_language == "Marwari":

        source_code = NIRUKTI_LANGUAGE_CODES["Marwari"]

        marwari_tokenizer.src_lang = source_code

        inputs = marwari_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )

        with torch.no_grad():

            translated_tokens = marwari_model.generate(
                **inputs,
                forced_bos_token_id=
                    marwari_tokenizer.convert_tokens_to_ids(
                        NIRUKTI_LANGUAGE_CODES["English"]
                    ),
                max_new_tokens=128
            )

        translated_text = marwari_tokenizer.decode(
            translated_tokens[0],
            skip_special_tokens=True
        )

        return translated_text.strip()


    # --------------------------------------------------------
    # Hindi / Marathi
    # NLLB-200
    # --------------------------------------------------------

    if source_language in NLLB_LANGUAGE_CODES:

        source_code = NLLB_LANGUAGE_CODES[source_language]

        translation_tokenizer.src_lang = source_code

        inputs = translation_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )

        with torch.no_grad():

            translated_tokens = translation_model.generate(
                **inputs,
                forced_bos_token_id=
                    translation_tokenizer.convert_tokens_to_ids(
                        "eng_Latn"
                    ),
                max_new_tokens=128
            )

        translated_text = translation_tokenizer.decode(
            translated_tokens[0],
            skip_special_tokens=True
        )

        return translated_text.strip()


    # --------------------------------------------------------
    # Unsupported language
    # --------------------------------------------------------

    raise ValueError(
        f"Unsupported source language: {source_language}"
    )

HEADING EMBEDDINGS

In [12]:
heading_texts = [
    chunk.get("heading", "")
    for chunk in chunks
]

heading_embeddings = embedding_model.encode(
    heading_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Heading embeddings created:", heading_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Heading embeddings created: (10, 768)


CREATE LANGUAGE-AWARE LEXICAL FEATURES

In [13]:
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


STOPWORDS = {
    "what", "is", "the", "a", "an",
    "are", "for", "of", "to", "in",
    "on", "and", "or", "how",
    "this", "that",

    "क्या", "है", "के", "का", "की",
    "और", "में", "से", "लिए",
    "कितना", "कितनी", "कौन",

    "काय", "आहे", "च्या", "चा",
    "ची", "चे", "आणि", "मध्ये",
    "साठी", "किती", "कोण"
}


def get_terms(text):

    tokens = re.findall(
        r"[\w]+",
        text.lower(),
        flags=re.UNICODE
    )

    return {
        token
        for token in tokens
        if token not in STOPWORDS
        and len(token) > 1
    }


def lexical_overlap(question, document):

    q_terms = get_terms(question)
    d_terms = get_terms(document)

    if not q_terms:
        return 0.0

    common = q_terms.intersection(d_terms)

    return len(common) / len(q_terms)


print("Lexical scoring ready.")

Lexical scoring ready.


DEFINE TEXT NORMALIZATION AND OVERLAP FEATURES

In [14]:
import re
import numpy as np


def normalize_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def get_words(text):

    text = normalize_text(text)

    return set(
        word
        for word in text.split()
        if len(word) > 1
        and word not in STOPWORDS
    )


def word_overlap(question, text):

    q_words = get_words(question)
    t_words = get_words(text)

    if not q_words:
        return 0.0

    common_words = q_words.intersection(t_words)

    return len(common_words) / len(q_words)


def phrase_overlap(question, text):

    q = normalize_text(question)
    t = normalize_text(text)

    if not q or not t:
        return 0.0

    q_words = q.split()

    if len(q_words) < 2:
        return 0.0

    phrases_found = 0
    total_phrases = 0

    for i in range(len(q_words) - 1):

        phrase = (
            q_words[i]
            + " "
            + q_words[i + 1]
        )

        total_phrases += 1

        if phrase in t:
            phrases_found += 1

    if total_phrases == 0:
        return 0.0

    return phrases_found / total_phrases

 IMPLEMENT MULTI-SIGNAL RAG RETRIEVER

In [15]:
import re
import numpy as np

QUESTION_INTENTS = {

    "age": [
        # English
        "age",
        "age limit",
        "age group",
        "years old",
        "how old",

        # Hindi
        "वय",
        "आयु",
        "आयु सीमा",

        # Marathi
        "वयोमर्यादा",

        # Marwari / Rajasthani-style vocabulary
        "उमर",
        "उमर सीमा",
        "उमर री",
        "साल",
        "कितणो",
        "कितणी"
    ],

    "income": [
        "income",
        "annual income",
        "family income",
        "income limit",

        "आय",
        "आय सीमा",
        "वार्षिक आय",

        "उत्पन्न",
        "उत्पन्न मर्यादा",

        # Marwari
        "आय गट",
        "कमाई",
        "आमदनी"
    ],

    "loan": [
        "loan",
        "loan amount",
        "amount",
        "maximum amount",

        "कर्ज",
        "कर्जाची रक्कम",
        "ऋण",
        "ऋण राशि",

        # Marwari
        "ज्यादा से ज्यादा ऋण रक़म",
        "ऋण री रक़म",
        "ऋण खातर",
        "खातर",
        "अर",
    ],

    "interest": [
        "interest",
        "interest rate",
        "rate of interest",
        "percentage",

        "ब्याज",
        "ब्याज दर",

        "व्याज",
        "व्याजदर",

        # Marwari
        "ब्याज रो दर",
        "ब्याज री दर",
    ],

    "repayment": [
        "repayment",
        "repayment period",
        "payable",

        "भुगतान",
        "भुगतान अवधि",

        "परतफेड",
        "परतफेड कालावधी",

        # Marwari
        "चुकावण रो समय",
        "चुकावण री अवधि",
    ],

    "moratorium": [
        "moratorium",
        "moratorium period",
        "waiting period",

        "स्थगन",
        "स्थगन अवधि",
        "मोरेटोरियम",

        # Marwari
        "मोरेटोरियम रो समय",
        "रोक रो समय",
    ],

    "eligibility": [
        "eligible",
        "eligibility",
        "eligibility criteria",
        "who can apply",

        "पात्र",
        "पात्रता",
        "कौन आवेदन कर सकता है",

        "पात्रता निकष",

        # Marwari
        "कुण पात्र",
        "कुण आवेदन",
        "पात्र है",
        "कुण ले सके"
    ],

    "documents": [
        "document",
        "documents",
        "required documents",
        "certificate",

        "दस्तावेज",
        "आवश्यक दस्तावेज",

        "कागदपत्रे",
        "आवश्यक कागदपत्रे",

        # Marwari
        "कागद",
        "कागजात",
        "दस्तावेज"
    ]
}

# ------------------------------------------------------------
# 2. Detect question intent
# ------------------------------------------------------------

def detect_question_intent(question):

    question = str(question).lower()

    detected = []

    for intent, keywords in QUESTION_INTENTS.items():

        for keyword in keywords:

            if keyword.lower() in question:

                detected.append(intent)
                break

    return detected


# ------------------------------------------------------------
# 3. Intent score for a document heading/body
# ------------------------------------------------------------

def intent_match_score(question, heading, context):

    intents = detect_question_intent(question)

    if not intents:
        return 0.0

    document_text = (
        str(heading) + " " + str(context)
    ).lower()

    matched = 0

    for intent in intents:

        keywords = QUESTION_INTENTS[intent]

        if any(
            keyword.lower() in document_text
            for keyword in keywords
        ):
            matched += 1

    return matched / len(intents)


# ------------------------------------------------------------
# 4. Numeric/entity preservation signal
# Useful for questions involving:
# age, percentage, money, years, limits, etc.
# ------------------------------------------------------------

def normalize_unicode_digits(text):

    if not text:
        return ""

    digit_map = str.maketrans(
        "०१२३४५६७८९",
        "0123456789"
    )

    return str(text).translate(digit_map)


def numeric_match_score(question, context):

    question = normalize_unicode_digits(question)
    context = normalize_unicode_digits(context)

    question_numbers = set(
        re.findall(
            r"\d+(?:\.\d+)?",
            str(question)
        )
    )

    context_numbers = set(
        re.findall(
            r"\d+(?:\.\d+)?",
            str(context)
        )
    )

    if not question_numbers:
        return 0.0

    common = question_numbers.intersection(
        context_numbers
    )

    return len(common) / len(question_numbers)


# ------------------------------------------------------------
# 5. Generic retrieval algorithm
# ------------------------------------------------------------

def custom_retrieve(
    original_question,
    english_query,
    top_k=5
):

    # --------------------------------------------------------
    # Query embedding
    # --------------------------------------------------------

    query_embedding = embedding_model.encode(
        english_query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # --------------------------------------------------------
    # Semantic similarity
    # --------------------------------------------------------

    semantic_scores = np.dot(
        document_embeddings,
        query_embedding
    )

    results = []

    # --------------------------------------------------------
    # Score every chunk
    # --------------------------------------------------------

    for i, chunk in enumerate(chunks):

        context = str(
            chunk.get("text", "")
        )

        heading = str(
            chunk.get("heading", "")
        )

        # ----------------------------------------------------
        # Semantic score
        # ----------------------------------------------------

        semantic_score = float(
            semantic_scores[i]
        )

        semantic_score = max(
            0.0,
            min(1.0, semantic_score)
        )

        # ----------------------------------------------------
        # Heading semantic similarity
        # ----------------------------------------------------

        heading_score = float(
            np.dot(
                heading_embeddings[i],
                query_embedding
            )
        )

        heading_score = max(
            0.0,
            min(1.0, heading_score)
        )

        # ----------------------------------------------------
        # Lexical body score
        # ----------------------------------------------------

        body_score = word_overlap(
            english_query,
            context
        )

        # ----------------------------------------------------
        # Heading lexical score
        # ----------------------------------------------------

        heading_lexical = word_overlap(
            english_query,
            heading
        )

        # ----------------------------------------------------
        # Phrase score
        # ----------------------------------------------------

        phrase_score = phrase_overlap(
            english_query,
            heading + " " + context
        )

        # ----------------------------------------------------
        # Intent score
        #
        # Use BOTH original question and English query.
        # This helps multilingual questions.
        # ----------------------------------------------------

        intent_score_english = intent_match_score(
            english_query,
            heading,
            context
        )

        intent_score_original = intent_match_score(
            original_question,
            heading,
            context
        )

        intent_score = max(
            intent_score_english,
            intent_score_original
        )

        # ----------------------------------------------------
        # Numeric match
        # ----------------------------------------------------

        numeric_score = numeric_match_score(
            english_query,
            context
        )

        # ----------------------------------------------------
        # OUR OWN RETRIEVAL ALGORITHM
        #
        # Semantic      = meaning
        # Heading       = section relevance
        # Body lexical  = direct word evidence
        # Heading lexical = exact section words
        # Phrase        = phrase-level evidence
        # Intent        = question type
        # Numeric       = numeric/entity evidence
        # ----------------------------------------------------

        final_score = (

            0.30 * semantic_score

            + 0.20 * heading_score

            + 0.10 * body_score

            + 0.10 * heading_lexical

            + 0.05 * phrase_score

            + 0.20 * intent_score

            + 0.05 * numeric_score
        )

        final_score = float(
            np.clip(
                final_score,
                0,
                1
            )
        )

        results.append({

            "chunk_id":
                chunk["chunk_id"],

            "heading":
                heading,

            "context":
                context,

            "semantic_score":
                round(
                    semantic_score,
                    4
                ),

            "heading_score":
                round(
                    heading_score,
                    4
                ),

            "body_score":
                round(
                    body_score,
                    4
                ),

            "heading_lexical":
                round(
                    heading_lexical,
                    4
                ),

            "phrase_score":
                round(
                    phrase_score,
                    4
                ),

            "intent_score":
                round(
                    intent_score,
                    4
                ),

            "numeric_score":
                round(
                    numeric_score,
                    4
                ),

            "retrieval_score":
                round(
                    final_score,
                    4
                )
        })

    # --------------------------------------------------------
    # Sort by our final score
    # --------------------------------------------------------

    results.sort(
        key=lambda x: x["retrieval_score"],
        reverse=True
    )

    return results[:top_k]

DEFINE MULTILINGUAL ANSWER TRANSLATION

In [16]:
def translate_answer_to_language(text, target_language):

    # English -> no translation
    if target_language == "English":
        return text

    # --------------------------------------------------------
    # ENGLISH -> MARWARI using Nirukti
    # --------------------------------------------------------
    if target_language == "Marwari":

        marwari_tokenizer.src_lang = "eng_Latn"

        inputs = marwari_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )

        with torch.no_grad():
            translated_tokens = marwari_model.generate(
                **inputs,
                forced_bos_token_id=
                    marwari_tokenizer.convert_tokens_to_ids(
                        "mwr_Deva"
                    ),
                max_new_tokens=128
            )

        translated_text = marwari_tokenizer.decode(
            translated_tokens[0],
            skip_special_tokens=True
        )

        return translated_text.strip()

    # --------------------------------------------------------
    # ENGLISH -> HINDI / MARATHI using NLLB
    # --------------------------------------------------------

    if target_language not in NLLB_LANGUAGE_CODES:
        raise ValueError(
            f"Unsupported target language: {target_language}"
        )

    target_code = NLLB_LANGUAGE_CODES[target_language]

    translation_tokenizer.src_lang = "eng_Latn"

    inputs = translation_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        translated_tokens = translation_model.generate(
            **inputs,
            forced_bos_token_id=
                translation_tokenizer.convert_tokens_to_ids(
                    target_code
                ),
            max_new_tokens=128
        )

    translated_text = translation_tokenizer.decode(
        translated_tokens[0],
        skip_special_tokens=True
    )

    return translated_text.strip()

DEFINE MULTILINGUAL ANSWER TRANSLATION

In [17]:
def final_answer_in_language(
    english_answer,
    language
):

    if language == "English":
        return english_answer

    return translate_answer_to_language(
        english_answer,
        language
    )

LOAD THE RAG ANSWER GENERATION MODEL

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "google/flan-t5-base"

print("Loading LLM...")

generator_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

generator_model = generator_model.to(device)
generator_model.eval()

print("LLM loaded successfully.")
print("Device:", device)

Loading LLM...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded successfully.
Device: cpu


DEFINE GROUNDED RAG ANSWER GENERATION

In [19]:
def generate_rag_answer(question, retrieved_evidence):
    """
    Generate a grounded answer from retrieved RAG evidence.
    Uses the LLM first and an evidence-based fallback if needed.
    """

    # ============================================================
    # STEP 1: PREPARE EVIDENCE
    # ============================================================

    if isinstance(retrieved_evidence, list):

        evidence_parts = []

        for item in retrieved_evidence:

            if isinstance(item, dict):
                text = item.get(
                    "context",
                    item.get("text", "")
                )
            else:
                text = str(item)

            if text:
                evidence_parts.append(text)

        context = "\n".join(evidence_parts)

    else:

        context = str(retrieved_evidence)

    # ============================================================
    # STEP 2: LIMIT CONTEXT
    # ============================================================

    context = context[:5000]

    # ============================================================
    # STEP 3: GROUNDED PROMPT
    # ============================================================

    prompt = f"""
You are a document question answering system.

Answer the question ONLY using the information present
in the provided context.

Do not use outside knowledge.

If the answer is present in the context, provide all
relevant factual information needed to answer the question.

Do not omit important numerical values, amounts,
percentages, conditions, or durations that are directly
present in the context.

If the answer is not present in the context, say:

Information not found in the provided context.

Question:
{question}

Context:
{context}

Answer:
"""

    # ============================================================
    # STEP 4: GENERATE USING LLM
    # ============================================================

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {
        key: value.to(generator_model.device)
        for key, value in inputs.items()
    }

    output = generator_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        num_beams=4
    )

    answer = generator_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    ).strip()

    # ============================================================
    # STEP 5: CLEAN ANSWER
    # ============================================================

    answer = answer.replace(
        "Answer:",
        ""
    ).strip()

    # ============================================================
    # STEP 6: OWN EVIDENCE-BASED FALLBACK
    # ============================================================

    bad_answers = [
        "information not found",
        "not found in the document",
        "not available",
        "cannot answer",
        "i don't know"
    ]

    needs_fallback = (
        len(answer) < 3
        or any(
            bad_text in answer.lower()
            for bad_text in bad_answers
        )
    )

    if needs_fallback:

        sentences = re.split(
            r"[.!?\n]",
            context
        )

        question_words = set(
            re.findall(
                r"\w+",
                question.lower(),
                flags=re.UNICODE
            )
        )

        best_sentence = ""
        best_score = 0

        for sentence in sentences:

            sentence = sentence.strip()

            if not sentence:
                continue

            sentence_words = set(
                re.findall(
                    r"\w+",
                    sentence.lower(),
                    flags=re.UNICODE
                )
            )

            lexical_score = len(
                question_words.intersection(
                    sentence_words
                )
            )

            # Give extra importance to numerical evidence
            question_numbers = set(
                re.findall(
                    r"\d+(?:\.\d+)?",
                    question
                )
            )

            sentence_numbers = set(
                re.findall(
                    r"\d+(?:\.\d+)?",
                    sentence
                )
            )

            numeric_score = len(
                question_numbers.intersection(
                    sentence_numbers
                )
            )

            total_score = (
                lexical_score
                + 2 * numeric_score
            )

            if total_score > best_score:

                best_score = total_score
                best_sentence = sentence

        if best_sentence:

            answer = best_sentence

        else:

            answer = (
                "Information not found in the "
                "provided context."
            )

    return answer

CREATE A REUSABLE QUERY CONVERSION INTERFACE

In [20]:
def convert_query_to_english(question, language=None):
    if language is None:
        language = "English"

    return translate_to_english(question, language)

MARWARI TRANSLATION MODEL

In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MARWARI_MODEL_NAME = "Anonym-050326/nirukti-translate-1.3b"

print("Loading Marwari translation model...")

marwari_tokenizer = AutoTokenizer.from_pretrained(
    MARWARI_MODEL_NAME
)

marwari_model = AutoModelForSeq2SeqLM.from_pretrained(
    MARWARI_MODEL_NAME
)

print("Marwari model loaded successfully!")

# Check whether the model supports Marwari
print(
    "mwr_Deva in tokenizer:",
    "mwr_Deva" in marwari_tokenizer.get_vocab()
)

Loading Marwari translation model...


config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.65k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 31.9MB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

Marwari model loaded successfully!
mwr_Deva in tokenizer: True


Multilingual normalization

In [22]:
import re
import numpy as np
from difflib import SequenceMatcher
from sklearn.metrics.pairwise import cosine_similarity


def normalize_unicode_digits(text):
    """
    Convert Unicode digits into standard English digits.

    Example:
        १६ -> 16
        ३२ -> 32
    """

    if not text:
        return ""

    digit_map = str.maketrans(
        "०१२३४५६७८९",
        "0123456789"
    )

    return str(text).translate(digit_map)


def normalize_eval_text(text):
    """
    Basic multilingual text normalization.
    """

    if not text:
        return ""

    text = normalize_unicode_digits(text)

    text = str(text).lower()

    # Normalize punctuation
    text = re.sub(r"[।,!?;:()\[\]{}]", " ", text)

    # Normalize multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_numbers(text):
    """
    Extract numerical facts from multilingual text.
    """

    text = normalize_unicode_digits(text)

    numbers = re.findall(
        r"\d+(?:\.\d+)?",
        text
    )

    return set(numbers)


def extract_ranges(text):
    """
    Extract numerical ranges.

    Handles examples such as:
        16 to 32
        16-32
        १६ ते ३२
        १६ से ३२
    """

    text = normalize_unicode_digits(text)

    pattern = r"""
        (\d+(?:\.\d+)?)
        \s*
        (?:to|-|–|—|से|ते)
        \s*
        (\d+(?:\.\d+)?)
    """

    matches = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE | re.VERBOSE
    )

    ranges = set()

    for start, end in matches:
        ranges.add(
            (
                float(start),
                float(end)
            )
        )

    return ranges


def token_set(text):
    """
    Return normalized multilingual tokens.
    """

    text = normalize_eval_text(text)

    return set(
        token
        for token in text.split()
        if token
    )


Multilingual fact matching

In [23]:
def numerical_fact_match(text_a, text_b):

    numbers_a = extract_numbers(text_a)
    numbers_b = extract_numbers(text_b)

    if not numbers_a:
        return 1.0

    if not numbers_b:
        return 0.0

    matched = numbers_a.intersection(numbers_b)

    return len(matched) / len(numbers_a)


def range_fact_match(text_a, text_b):

    ranges_a = extract_ranges(text_a)
    ranges_b = extract_ranges(text_b)

    if not ranges_a:
        return 1.0

    if not ranges_b:
        return 0.0

    matched = ranges_a.intersection(ranges_b)

    return len(matched) / len(ranges_a)


def multilingual_fact_match(text_a, text_b):

    number_score = numerical_fact_match(
        text_a,
        text_b
    )

    range_score = range_fact_match(
        text_a,
        text_b
    )

    final_score = (
        0.50 * number_score +
        0.50 * range_score
    )

    return round(
        float(final_score),
        4
    )


print("Multilingual fact matching functions loaded.")

Multilingual fact matching functions loaded.


ATOMIC FACT COVERAGE ALGORITHM

In [24]:
def extract_atomic_facts(text):

    if not text:
        return []

    text = normalize_unicode_digits(
        str(text)
    )

    facts = []

    # --------------------------------------------------------
    # Monetary values
    # --------------------------------------------------------

    money_matches = re.findall(
        r"(?:₹|rs\.?|rupees?|रु|रुपये?)?\s*"
        r"(\d+(?:\.\d+)?)\s*"
        r"(?:lakh|lakhs|लाख)",
        text,
        flags=re.IGNORECASE
    )

    for value in money_matches:

        facts.append(
            ("money", float(value))
        )

    # --------------------------------------------------------
    # Duration values
    # --------------------------------------------------------

    duration_matches = re.findall(
        r"(\d+(?:\.\d+)?)\s*"
        r"(?:years?|year|months?|month|"
        r"वर्ष|वर्षे|साल|महीने)",
        text,
        flags=re.IGNORECASE
    )

    for value in duration_matches:

        facts.append(
            ("duration", float(value))
        )

    # --------------------------------------------------------
    # Percentage values
    # --------------------------------------------------------

    percentage_matches = re.findall(
        r"(\d+(?:\.\d+)?)\s*%",
        text
    )

    for value in percentage_matches:

        facts.append(
            ("percentage", float(value))
        )

    return facts


def atomic_fact_coverage(
    reference,
    candidate
):

    reference_facts = extract_atomic_facts(
        reference
    )

    candidate_facts = extract_atomic_facts(
        candidate
    )

    if not reference_facts:
        return 1.0

    matched = 0

    for ref_type, ref_value in reference_facts:

        for cand_type, cand_value in candidate_facts:

            if (
                ref_type == cand_type
                and ref_value == cand_value
            ):

                matched += 1
                break

    score = (
        matched /
        len(reference_facts)
    )

    return round(
        float(score),
        4
    )


print(
    "Own atomic fact coverage algorithm loaded."
)

Own atomic fact coverage algorithm loaded.


MULTILINGUAL SEMANTIC + FACTUAL EVIDENCE MATCHING

In [25]:
def multilingual_semantic_score(text_a, text_b):

    if not text_a or not text_b:
        return 0.0

    embedding_a = embedding_model.encode(
        [str(text_a)],
        normalize_embeddings=True
    )

    embedding_b = embedding_model.encode(
        [str(text_b)],
        normalize_embeddings=True
    )

    score = cosine_similarity(
        embedding_a,
        embedding_b
    )[0][0]

    return float(
        np.clip(score, 0.0, 1.0)
    )


def multilingual_evidence_score(
    answer,
    context
):

    if not answer or not context:
        return 0.0

    # --------------------------------------------------------
    # 1. Numerical / range factual support
    # --------------------------------------------------------

    fact_score = multilingual_fact_match(
        answer,
        context
    )

    # --------------------------------------------------------
    # 2. Multilingual semantic support
    # --------------------------------------------------------

    semantic_score = multilingual_semantic_score(
        answer,
        context
    )

    # --------------------------------------------------------
    # 3. Final evidence score
    # --------------------------------------------------------

    final_score = (
        0.60 * fact_score +
        0.40 * semantic_score
    )

    return round(
        float(np.clip(final_score, 0.0, 1.0)),
        4
    )


print("Multilingual evidence scoring loaded.")

Multilingual evidence scoring loaded.


Answer Similarity

In [26]:
def answer_similarity_multilingual(
    expected_answer,
    generated_answer
):

    if not expected_answer or not generated_answer:
        return 0.0

    # --------------------------------------------------------
    # 1. Multilingual semantic similarity
    # --------------------------------------------------------

    semantic_score = multilingual_semantic_score(
        expected_answer,
        generated_answer
    )

    # --------------------------------------------------------
    # 2. Numerical / range factual similarity
    # --------------------------------------------------------

    fact_score = multilingual_fact_match(
        expected_answer,
        generated_answer
    )

    # --------------------------------------------------------
    # 3. Normalized text similarity
    # --------------------------------------------------------

    text_score = SequenceMatcher(
        None,
        normalize_eval_text(expected_answer),
        normalize_eval_text(generated_answer)
    ).ratio()

    # --------------------------------------------------------
    # Final multilingual similarity
    # --------------------------------------------------------

    atomic_score = atomic_fact_coverage(
    expected_answer,
    generated_answer
    )

    final_score = (
    0.40 * semantic_score +
    0.30 * fact_score +
    0.25 * atomic_score +
    0.05 * text_score
    )

    return round(
        float(np.clip(final_score, 0.0, 1.0)),
        4
    )

In [27]:
import pandas as pd

Answer Correctness

In [28]:
def answer_correctness_multilingual(
    expected_answer,
    generated_answer
):

    if not expected_answer or not generated_answer:
        return 0.0

    fact_score = multilingual_fact_match(
        expected_answer,
        generated_answer
    )

    semantic_score = multilingual_semantic_score(
        expected_answer,
        generated_answer
    )

    text_score = SequenceMatcher(
        None,
        normalize_eval_text(expected_answer),
        normalize_eval_text(generated_answer)
    ).ratio()

    atomic_score = atomic_fact_coverage(
    expected_answer,
    generated_answer
    )

    final_score = (
    0.40 * fact_score +
    0.30 * atomic_score +
    0.25 * semantic_score +
    0.05 * text_score
    )

    return round(
        float(np.clip(final_score, 0.0, 1.0)),
        4
    )

Answer Relevancy

In [29]:
ANSWER_TYPE_PATTERNS = {

    "age": [
        r"\d+\s*(?:to|-|–|—|से|ते)\s*\d+",
        r"\d+\s*(?:years?|वर्ष|साल|वय|उमर)"
    ],

    "income": [
        r"(?:₹|rs\.?|रु|रुपये?)\s*\d+",
        r"\d+\s*(?:lakh|lakhs|लाख)"
    ],

    "loan": [
        r"(?:₹|rs\.?|रु|रुपये?)\s*\d+",
        r"\d+\s*(?:lakh|lakhs|लाख)"
    ],

    "interest": [
        r"\d+\s*%",
        r"\d+\s*(?:percent|प्रतिशत)"
    ],

    "repayment": [
        r"\d+\s*(?:years?|months?|वर्ष|साल|महीने)"
    ],

    "moratorium": [
        r"\d+\s*(?:months?|years?|महीने|वर्ष|साल)"
    ]
}


def answer_type_compatibility(question, answer):

    intents = detect_question_intent(question)

    if not intents:
        return 0.5

    matched = 0

    for intent in intents:

        patterns = ANSWER_TYPE_PATTERNS.get(
            intent,
            []
        )

        for pattern in patterns:

            if re.search(
                pattern,
                normalize_unicode_digits(
                    str(answer).lower()
                )
            ):
                matched += 1
                break

    return min(
        1.0,
        matched / len(intents)
    )


def question_answer_semantic_score(
    question,
    answer
):

    q_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    a_embedding = embedding_model.encode(
        [answer],
        normalize_embeddings=True
    )

    score = cosine_similarity(
        q_embedding,
        a_embedding
    )[0][0]

    return max(
        0.0,
        min(1.0, float(score))
    )


def answer_evidence_alignment(
    answer,
    context
):

    if not answer or not context:
        return 0.0

    # Numerical/range evidence
    fact_score = multilingual_fact_match(
        answer,
        context
    )

    # Multilingual semantic evidence
    answer_embedding = embedding_model.encode(
        [answer],
        normalize_embeddings=True
    )

    context_embedding = embedding_model.encode(
        [context],
        normalize_embeddings=True
    )

    semantic_score = cosine_similarity(
        answer_embedding,
        context_embedding
    )[0][0]

    semantic_score = max(
        0.0,
        min(1.0, float(semantic_score))
    )

    # Evidence score
    final_score = (
        0.70 * fact_score +
        0.30 * semantic_score
    )

    return round(
        float(final_score),
        4
    )


def answer_directness_score(answer):

    tokens = token_set(answer)

    if not tokens:
        return 0.0

    if len(tokens) <= 30:
        return 1.0

    if len(tokens) <= 60:
        return 0.85

    if len(tokens) <= 100:
        return 0.70

    return 0.50


def custom_answer_relevancy(
    question,
    answer,
    context
):

    semantic_score = question_answer_semantic_score(
        question,
        answer
    )

    type_score = answer_type_compatibility(
        question,
        answer
    )

    evidence_score = multilingual_evidence_score(
    answer,
    context
    )

    directness_score = answer_directness_score(
        answer
    )

    final_score = (
        0.20 * semantic_score +
        0.25 * type_score +
        0.40 * evidence_score +
        0.15 * directness_score
    )

    return round(
        float(final_score),
        4
    )

Completeness

In [30]:
def semantic_coverage(
    expected_answer,
    generated_answer
):

    expected_embedding = embedding_model.encode(
        [expected_answer],
        normalize_embeddings=True
    )

    generated_embedding = embedding_model.encode(
        [generated_answer],
        normalize_embeddings=True
    )

    score = cosine_similarity(
        expected_embedding,
        generated_embedding
    )[0][0]

    return max(
        0.0,
        min(1.0, float(score))
    )


def completeness_score_multilingual(
    expected_answer,
    generated_answer,
    context
):

    if not expected_answer or not generated_answer:
        return 0.0

    # --------------------------------------------------------
    # 1. Expected fact coverage
    # --------------------------------------------------------

    fact_score = multilingual_fact_match(
        expected_answer,
        generated_answer
    )

    # --------------------------------------------------------
    # 2. Semantic coverage
    # --------------------------------------------------------

    semantic_score = semantic_coverage(
        expected_answer,
        generated_answer
    )

    # --------------------------------------------------------
    # 3. Evidence coverage
    # --------------------------------------------------------

    evidence_score = answer_evidence_alignment(
        generated_answer,
        context
    )

    final_score = (
        0.45 * fact_score +
        0.30 * semantic_score +
        0.25 * evidence_score
    )

    return round(
        float(final_score),
        4
    )

Factual Support

In [31]:
def factual_support_multilingual(
    generated_answer,
    context
):

    if not generated_answer or not context:
        return 0.0

    evidence_score = multilingual_evidence_score(
        generated_answer,
        context
    )

    atomic_score = atomic_fact_coverage(
        context,
        generated_answer
    )

    final_score = (
        0.60 * evidence_score +
        0.40 * atomic_score
    )

    return round(
        float(np.clip(
            final_score,
            0.0,
            1.0
        )),
        4
    )

FactCC-style

In [32]:
def custom_factcc_style(
    generated_answer,
    context
):

    if not generated_answer or not context:
        return 0.0

    number_score = numerical_fact_match(
        generated_answer,
        context
    )

    range_score = range_fact_match(
        generated_answer,
        context
    )

    semantic_score = multilingual_semantic_score(
        generated_answer,
        context
    )

    final_score = (
        0.30 * number_score +
        0.30 * range_score +
        0.40 * semantic_score
    )

    return round(
        float(np.clip(final_score, 0.0, 1.0)),
        4
    )

SummaC

In [33]:
def split_claims(text):

    if not text:
        return []

    text = str(text)

    # Handle sentence endings in English and Indian languages
    text = text.replace("।", ".")
    text = text.replace("!", ".")
    text = text.replace("?", ".")

    claims = [
        claim.strip()
        for claim in text.split(".")
        if claim.strip()
    ]

    return claims


def split_evidence(context):

    if not context:
        return []

    context = str(context)

    # Normalize common sentence separators
    context = context.replace("।", ".")
    context = context.replace("!", ".")
    context = context.replace("?", ".")

    evidence_units = [
        sentence.strip()
        for sentence in context.split(".")
        if sentence.strip()
    ]

    # If sentence splitting produces nothing,
    # use the complete context as one evidence unit.
    if not evidence_units:
        evidence_units = [context.strip()]

    return evidence_units


def custom_summac_style(
    generated_answer,
    context
):

    # --------------------------------------------------------
    # 1. Extract claims from generated answer
    # --------------------------------------------------------

    claims = split_claims(
        generated_answer
    )

    if not claims:
        return 0.0

    # --------------------------------------------------------
    # 2. Split retrieved context into evidence units
    # --------------------------------------------------------

    evidence_units = split_evidence(
        context
    )

    if not evidence_units:
        return 0.0

    claim_scores = []

    # --------------------------------------------------------
    # 3. Evaluate every generated claim
    # --------------------------------------------------------

    for claim in claims:

        best_score = 0.0

        for evidence in evidence_units:

            # ------------------------------------------------
            # Semantic + numerical evidence support
            # ------------------------------------------------

            evidence_score = multilingual_evidence_score(
                claim,
                evidence
            )

            # ------------------------------------------------
            # Claim-level atomic fact coverage
            #

            atomic_score = atomic_fact_coverage(
                claim,
                evidence
            )

            # ------------------------------------------------
            # Combine evidence signals
            # ------------------------------------------------

            combined_score = (
                0.70 * evidence_score +
                0.30 * atomic_score
            )

            # Keep the strongest evidence for this claim
            if combined_score > best_score:
                best_score = combined_score

        claim_scores.append(
            best_score
        )

    # --------------------------------------------------------
    # 4. Average support across all generated claims
    # --------------------------------------------------------

    final_score = np.mean(
        claim_scores
    )

    return round(
        float(
            np.clip(
                final_score,
                0.0,
                1.0
            )
        ),
        4
    )


print(
    "Custom SummaC-style claim-level evaluation loaded."
)

Custom SummaC-style claim-level evaluation loaded.


QAGS-

In [34]:
def custom_qags_style(
    generated_answer,
    context
):

    if not generated_answer or not context:
        return 0.0

    fact_agreement = multilingual_fact_match(
        generated_answer,
        context
    )

    semantic_agreement = multilingual_evidence_score(
            generated_answer,
            context
    )

    final_score = (
        0.60 * fact_agreement +
        0.40 * semantic_agreement
    )

    return round(
        float(final_score),
        4
    )

DAE

In [35]:
def custom_dae_style(
    generated_answer,
    context
):

    if not generated_answer or not context:
        return 0.0

    fact_support = multilingual_fact_match(
        generated_answer,
        context
    )

    semantic_support = multilingual_evidence_score(
            generated_answer,
            context
    )

    final_score = (
        0.60 * fact_support +
        0.40 * semantic_support
    )

    return round(
        float(final_score),
        4
    )

BLEURT

In [36]:
def custom_bleurt_style(
    reference,
    candidate
):

    if not reference or not candidate:
        return 0.0

    # --------------------------------------------------------
    # 1. Multilingual semantic similarity
    # --------------------------------------------------------

    ref_embedding = embedding_model.encode(
        [reference],
        normalize_embeddings=True
    )

    candidate_embedding = embedding_model.encode(
        [candidate],
        normalize_embeddings=True
    )

    semantic_score = cosine_similarity(
        ref_embedding,
        candidate_embedding
    )[0][0]

    semantic_score = max(
        0.0,
        min(1.0, float(semantic_score))
    )

    # --------------------------------------------------------
    # 2. Factual agreement
    # --------------------------------------------------------

    fact_score = multilingual_fact_match(
        reference,
        candidate
    )

    # --------------------------------------------------------
    # 3. Text similarity
    # --------------------------------------------------------

    text_score = SequenceMatcher(
        None,
        normalize_eval_text(reference),
        normalize_eval_text(candidate)
    ).ratio()

    # --------------------------------------------------------
    # 4. Final score
    # --------------------------------------------------------

    final_score = (
        0.55 * semantic_score +
        0.30 * fact_score +
        0.15 * text_score
    )

    return round(
        float(final_score),
        4
    )

In [40]:
import pandas as pd
from IPython.display import display


def run_language_evaluation(
    query,
    grounded_answer,
    language
):

    print("=" * 80)
    print(f"LANGUAGE: {language}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Convert user query to English for retrieval
    # --------------------------------------------------------

    english_query = convert_query_to_english(
        query,
        language
    )

    print("\nOriginal Query:")
    print(query)

    print("\nEnglish Query Used for Retrieval:")
    print(english_query)

    # --------------------------------------------------------
    # 2. Retrieve relevant chunks
    # --------------------------------------------------------

    retrieved_results = custom_retrieve(
        query,
        english_query,
        top_k=5
    )

    print("\n" + "-" * 80)
    print("RETRIEVED CONTEXT")
    print("-" * 80)

    if not retrieved_results:
        print("No relevant context retrieved.")
        return None

    for i, item in enumerate(
        retrieved_results,
        start=1
    ):

        print(f"\nChunk {i}")
        print(f"Chunk ID: {item.get('chunk_id', 'N/A')}")
        print(f"Heading: {item.get('heading', 'N/A')}")
        print(
            f"Retrieval Score: "
            f"{item.get('retrieval_score', 0):.4f}"
        )

        print("Context:")
        print(item.get("context", ""))

    # --------------------------------------------------------
    # 3. Combine retrieved evidence
    # --------------------------------------------------------

    retrieved_context = "\n\n".join(
        item.get("context", "")
        for item in retrieved_results
    )

    # --------------------------------------------------------
    # 4. Generate grounded English answer
    # --------------------------------------------------------

    english_answer = generate_rag_answer(
        english_query,
        retrieved_results
    )

    # --------------------------------------------------------
    # 5. Convert answer into requested language
    # --------------------------------------------------------

    generated_answer = final_answer_in_language(
        english_answer,
        language
    )

    print("\n" + "-" * 80)
    print("GROUNDED / REFERENCE ANSWER")
    print("-" * 80)
    print(grounded_answer)

    print("\n" + "-" * 80)
    print("GENERATED ANSWER")
    print("-" * 80)
    print(generated_answer)

    # ========================================================
    # 6. EVALUATION METRICS
    # ========================================================

    # --------------------------------------------------------
    # Answer Similarity
    # --------------------------------------------------------

    similarity_score = answer_similarity_multilingual(
        grounded_answer,
        generated_answer
    )

    # --------------------------------------------------------
    # Answer Correctness
    # --------------------------------------------------------

    correctness_score = answer_correctness_multilingual(
        grounded_answer,
        generated_answer
    )

    # --------------------------------------------------------
    # Answer Relevancy
    # --------------------------------------------------------

    relevancy_score = custom_answer_relevancy(
        query,
        generated_answer,
        retrieved_context
    )

    # --------------------------------------------------------
    # Completeness
    # --------------------------------------------------------

    completeness_score = completeness_score_multilingual(
        grounded_answer,
        generated_answer,
        retrieved_context
    )

    # --------------------------------------------------------
    # Factual Support
    # --------------------------------------------------------

    factual_support_score = factual_support_multilingual(
        generated_answer,
        retrieved_context
    )

    # --------------------------------------------------------
    # Custom FactCC-style
    # --------------------------------------------------------

    factcc_score = custom_factcc_style(
        generated_answer,
        retrieved_context
    )

    # --------------------------------------------------------
    # Custom SummaC-style
    # --------------------------------------------------------

    summac_score = custom_summac_style(
        generated_answer,
        retrieved_context
    )

    # --------------------------------------------------------
    # Custom QAGS-style
    # --------------------------------------------------------

    qags_score = custom_qags_style(
        generated_answer,
        retrieved_context
    )

    # --------------------------------------------------------
    # Custom DAE-style
    # --------------------------------------------------------

    dae_score = custom_dae_style(
        generated_answer,
        retrieved_context
    )

    # --------------------------------------------------------
    # Custom BLEURT-style
    # --------------------------------------------------------

    bleurt_score = custom_bleurt_style(
        grounded_answer,
        generated_answer
    )

    # ========================================================
    # 7. STORE ALL METRICS
    # ========================================================

    metric_scores = {

        "Answer Similarity":
            similarity_score,

        "Answer Correctness":
            correctness_score,

        "Answer Relevancy":
            relevancy_score,

        "Completeness":
            completeness_score,

        "Factual Support":
            factual_support_score,

        "Custom FactCC-style":
            factcc_score,

        "Custom SummaC-style":
            summac_score,

        "Custom QAGS-style":
            qags_score,

        "Custom DAE-style":
            dae_score,

        "Custom BLEURT-style":
            bleurt_score
    }

    # --------------------------------------------------------
    # 8. Calculate overall score
    # --------------------------------------------------------

    overall_score = sum(
        metric_scores.values()
    ) / len(metric_scores)

    overall_score = round(
        float(overall_score),
        4
    )

    # ========================================================
    # 9. Display metric table
    # ========================================================

    print("\n" + "=" * 80)
    print("EVALUATION METRICS")
    print("=" * 80)

    metrics_df = pd.DataFrame(
        {
            "Metric": list(
                metric_scores.keys()
            ),
            "Score": list(
                metric_scores.values()
            )
        }
    )

    display(metrics_df)

    # --------------------------------------------------------
    # 10. Overall score
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print(
        f"OVERALL SCORE ({language}): "
        f"{overall_score:.4f}"
    )
    print("=" * 80)

    # --------------------------------------------------------
    # 11. Return results
    # --------------------------------------------------------

    return {

        "language": language,

        "query": query,

        "english_query": english_query,

        "grounded_answer": grounded_answer,

        "english_answer": english_answer,

        "generated_answer": generated_answer,

        "retrieved_context": retrieved_context,

        "retrieved_results": retrieved_results,

        "metrics": metric_scores,

        "overall_score": overall_score,

        "metrics_df": metrics_df
    }


# ============================================================
# ENGLISH TEST CASE
# ============================================================

english_query = "What is the maximum loan amount for professional postgraduate courses in India and for courses abroad and what is the maximum course duration?"

english_grounded_answer = (
    "For professional postgraduate courses in India the maximum loan amount is Rs. 20.00 lakhs with a maximum course duration of 5 years.For courses abroad, the maximum loan amount is Rs. 30.00 lakhs, with a maximum course duration of 5 years."
)


english_result = run_language_evaluation(
    query=english_query,
    grounded_answer=english_grounded_answer,
    language="English"
)

LANGUAGE: English

Original Query:
What is the maximum loan amount for professional postgraduate courses in India and for courses abroad and what is the maximum course duration?

English Query Used for Retrieval:
What is the maximum loan amount for professional postgraduate courses in India and for courses abroad and what is the maximum course duration?

--------------------------------------------------------------------------------
RETRIEVED CONTEXT
--------------------------------------------------------------------------------

Chunk 1
Chunk ID: 4
Heading: Loan Amount
Retrieval Score: 0.5699
Context:
Loan Amount
Up to Rs. 20.00 Lakhs for „Professional Post Graduate Courses‟ with a maximum duration of 5 years @ Rs. 4.00 Lakhs per annum in India & Up to Rs.30.00 Lakhs for „Courses Abroad‟ with a maximum duration of 5 years @ Rs. 6.00 Lakhs per annum.

Chunk 2
Chunk ID: 0
Heading: General Information
Retrieval Score: 0.4474
Context:
General Information
Educational Loan Scheme Rajastha

,Metric,Score
0,Answer Similarity,0.7897
1,Answer Correctness,0.7822
2,Answer Relevancy,0.8610
3,Completeness,0.8659
4,Factual Support,0.8497
5,Custom FactCC-style,0.9162
6,Custom SummaC-style,0.9834
7,Custom QAGS-style,0.9665
8,Custom DAE-style,0.9665
9,Custom BLEURT-style,0.7505



OVERALL SCORE (English): 0.8732


In [41]:
hindi_query = "भारत में व्यावसायिक स्नातकोत्तर पाठ्यक्रमों और विदेश में पाठ्यक्रमों के लिए अधिकतम ऋण राशि क्या है और अधिकतम पाठ्यक्रम अवधि कितनी है?"

hindi_grounded_answer = (
    "भारत में व्यावसायिक स्नातकोत्तर पाठ्यक्रमों के लिए अधिकतम ऋण राशि 20 लाख रुपये है और अधिकतम अवधि 5 वर्ष है। विदेश में पाठ्यक्रमों के लिए अधिकतम ऋण राशि 30 लाख रुपये है और अधिकतम अवधि 5 वर्ष है।"
)


hindi_result = run_language_evaluation(
    query=hindi_query,
    grounded_answer=hindi_grounded_answer,
    language="Hindi"
)

LANGUAGE: Hindi


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original Query:
भारत में व्यावसायिक स्नातकोत्तर पाठ्यक्रमों और विदेश में पाठ्यक्रमों के लिए अधिकतम ऋण राशि क्या है और अधिकतम पाठ्यक्रम अवधि कितनी है?

English Query Used for Retrieval:
What is the maximum loan amount for vocational postgraduate courses in India and overseas courses and what is the maximum course duration?

--------------------------------------------------------------------------------
RETRIEVED CONTEXT
--------------------------------------------------------------------------------

Chunk 1
Chunk ID: 4
Heading: Loan Amount
Retrieval Score: 0.5409
Context:
Loan Amount
Up to Rs. 20.00 Lakhs for „Professional Post Graduate Courses‟ with a maximum duration of 5 years @ Rs. 4.00 Lakhs per annum in India & Up to Rs.30.00 Lakhs for „Courses Abroad‟ with a maximum duration of 5 years @ Rs. 6.00 Lakhs per annum.

Chunk 2
Chunk ID: 0
Heading: General Information
Retrieval Score: 0.4189
Context:
General Information
Educational Loan Scheme Rajasthan Minority Finance & Developmen

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--------------------------------------------------------------------------------
GROUNDED / REFERENCE ANSWER
--------------------------------------------------------------------------------
भारत में व्यावसायिक स्नातकोत्तर पाठ्यक्रमों के लिए अधिकतम ऋण राशि 20 लाख रुपये है और अधिकतम अवधि 5 वर्ष है। विदेश में पाठ्यक्रमों के लिए अधिकतम ऋण राशि 30 लाख रुपये है और अधिकतम अवधि 5 वर्ष है।

--------------------------------------------------------------------------------
GENERATED ANSWER
--------------------------------------------------------------------------------
व्यावसायिक पोस्ट ग्रेजुएट पाठ्यक्रमों के लिए अधिकतम 5 वर्ष की अवधि के साथ भारत में प्रतिवर्ष 4.00 लाख रुपये और विदेशों में अधिकतम 5 वर्ष की अवधि के लिए 30.00 लाख रुपये।

EVALUATION METRICS


,Metric,Score
0,Answer Similarity,0.7666
1,Answer Correctness,0.7389
2,Answer Relevancy,0.9071
3,Completeness,0.7926
4,Factual Support,0.7654
5,Custom FactCC-style,0.8868
6,Custom SummaC-style,0.8752
7,Custom QAGS-style,0.9547
8,Custom DAE-style,0.9547
9,Custom BLEURT-style,0.7657



OVERALL SCORE (Hindi): 0.8408


In [42]:
marathi_query = "भारतातील व्यावसायिक पदव्युत्तर अभ्यासक्रमांसाठी आणि परदेशातील अभ्यासक्रमांसाठी कमाल कर्ज रक्कम किती आहे आणि कमाल अभ्यासक्रम कालावधी किती आहे?"

marathi_grounded_answer = (
      "भारतातील व्यावसायिक पदव्युत्तर अभ्यासक्रमांसाठी कमाल कर्ज रक्कम 20 लाख रुपये आहे आणि कमाल अभ्यासक्रम कालावधी 5 वर्षे आहे. परदेशातील अभ्यासक्रमांसाठी कमाल कर्ज रक्कम 30 लाख रुपये आहे आणि कमाल कालावधी 5 वर्षे आहे."
)


marathi_result = run_language_evaluation(
    query=marathi_query,
    grounded_answer=marathi_grounded_answer,
    language="Marathi"
)

LANGUAGE: Marathi


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original Query:
भारतातील व्यावसायिक पदव्युत्तर अभ्यासक्रमांसाठी आणि परदेशातील अभ्यासक्रमांसाठी कमाल कर्ज रक्कम किती आहे आणि कमाल अभ्यासक्रम कालावधी किती आहे?

English Query Used for Retrieval:
What is the maximum loan amount for vocational postgraduate courses in India and overseas courses and what is the maximum duration of the course?

--------------------------------------------------------------------------------
RETRIEVED CONTEXT
--------------------------------------------------------------------------------

Chunk 1
Chunk ID: 4
Heading: Loan Amount
Retrieval Score: 0.5446
Context:
Loan Amount
Up to Rs. 20.00 Lakhs for „Professional Post Graduate Courses‟ with a maximum duration of 5 years @ Rs. 4.00 Lakhs per annum in India & Up to Rs.30.00 Lakhs for „Courses Abroad‟ with a maximum duration of 5 years @ Rs. 6.00 Lakhs per annum.

Chunk 2
Chunk ID: 0
Heading: General Information
Retrieval Score: 0.4231
Context:
General Information
Educational Loan Scheme Rajasthan Minority Finan

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--------------------------------------------------------------------------------
GROUNDED / REFERENCE ANSWER
--------------------------------------------------------------------------------
भारतातील व्यावसायिक पदव्युत्तर अभ्यासक्रमांसाठी कमाल कर्ज रक्कम 20 लाख रुपये आहे आणि कमाल अभ्यासक्रम कालावधी 5 वर्षे आहे. परदेशातील अभ्यासक्रमांसाठी कमाल कर्ज रक्कम 30 लाख रुपये आहे आणि कमाल कालावधी 5 वर्षे आहे.

--------------------------------------------------------------------------------
GENERATED ANSWER
--------------------------------------------------------------------------------
व्यावसायिक पदव्युत्तर अभ्यासक्रमांसाठी जास्तीत जास्त 20 लाख रुपये 5 वर्षांच्या कालावधीत. भारतात दरवर्षी 4.00 लाख रुपये आणि परदेशात जास्तीत जास्त 5 वर्षांच्या कालावधीत 6.00 लाख रुपये दरवर्षी 30.00 लाख रुपये पर्यंत.

EVALUATION METRICS


,Metric,Score
0,Answer Similarity,0.8611
1,Answer Correctness,0.8652
2,Answer Relevancy,0.8920
3,Completeness,0.8646
4,Factual Support,0.9064
5,Custom FactCC-style,0.8995
6,Custom SummaC-style,0.8320
7,Custom QAGS-style,0.9598
8,Custom DAE-style,0.9598
9,Custom BLEURT-style,0.7733



OVERALL SCORE (Marathi): 0.8814


In [43]:
marwari_query = "भारत में प्रोफेशनल पोस्ट ग्रेजुएट कोर्स खातर ज्यादा से ज्यादा ऋण रक़म कितणी है, अर विदेश रो कोर्स खातर कितणी है, अर कोर्स री ज्यादा से ज्यादा अवधि कितणी है?"

marwari_grounded_answer = (
    "भारत में प्रोफेशनल पोस्ट ग्रेजुएट कोर्स खातर ज्यादा से ज्यादा ऋण रक़म 20 लाख है अर कोर्स री ज्यादा से ज्यादा अवधि 5 साल है। विदेश में कोर्स खातर ज्यादा से ज्यादा ऋण रक़म 30 लाख है अर अवधि 5 साल है।"
)


marwari_result = run_language_evaluation(
    query=marwari_query,
    grounded_answer=marwari_grounded_answer,
    language="Marwari"
)

LANGUAGE: Marwari


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Original Query:
भारत में प्रोफेशनल पोस्ट ग्रेजुएट कोर्स खातर ज्यादा से ज्यादा ऋण रक़म कितणी है, अर विदेश रो कोर्स खातर कितणी है, अर कोर्स री ज्यादा से ज्यादा अवधि कितणी है?

English Query Used for Retrieval:
What is the maximum loan amount for a professional postgraduate course in India, and what is the maximum loan amount for a foreign course, and what is the maximum duration of the course?

--------------------------------------------------------------------------------
RETRIEVED CONTEXT
--------------------------------------------------------------------------------

Chunk 1
Chunk ID: 4
Heading: Loan Amount
Retrieval Score: 0.5588
Context:
Loan Amount
Up to Rs. 20.00 Lakhs for „Professional Post Graduate Courses‟ with a maximum duration of 5 years @ Rs. 4.00 Lakhs per annum in India & Up to Rs.30.00 Lakhs for „Courses Abroad‟ with a maximum duration of 5 years @ Rs. 6.00 Lakhs per annum.

Chunk 2
Chunk ID: 0
Heading: General Information
Retrieval Score: 0.4415
Context:
General Info

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--------------------------------------------------------------------------------
GROUNDED / REFERENCE ANSWER
--------------------------------------------------------------------------------
भारत में प्रोफेशनल पोस्ट ग्रेजुएट कोर्स खातर ज्यादा से ज्यादा ऋण रक़म 20 लाख है अर कोर्स री ज्यादा से ज्यादा अवधि 5 साल है। विदेश में कोर्स खातर ज्यादा से ज्यादा ऋण रक़म 30 लाख है अर अवधि 5 साल है।

--------------------------------------------------------------------------------
GENERATED ANSWER
--------------------------------------------------------------------------------
प्रोफेशनल पोस्ट ग्रेजुएट कोर्स के लिये २०.०० लाख रुपए, अधिकतम अवधि ५ वर्ष, भारत में ४.०० लाख रुपए प्रति वर्ष और विदेश में कोर्स के लिये ३०.०० लाख रुपए, अधिकतम अवधि ५ वर्ष, प्रति वर्ष ६.०० लाख रुपए।

EVALUATION METRICS


,Metric,Score
0,Answer Similarity,0.8161
1,Answer Correctness,0.8048
2,Answer Relevancy,0.9121
3,Completeness,0.7870
4,Factual Support,0.9058
5,Custom FactCC-style,0.8985
6,Custom SummaC-style,0.8659
7,Custom QAGS-style,0.9594
8,Custom DAE-style,0.9594
9,Custom BLEURT-style,0.7438



OVERALL SCORE (Marwari): 0.8653
